# Lily 1.5B GRPO Model Benchmark Evaluation — Modal A100
**Evaluates trained Lily-1.5B (GRPO model) using `lm-eval` harness**

Runs multi-fewshot benchmark evaluations (`gsm8k`, `arc_challenge`, `hellaswag`) on `abhinav0231/Lily-1.5B` using Modal A100 GPU with BFloat16 precision and batch size 16.

## Cell 1 — Install Dependencies (Modal %uv optimized)

In [1]:
# ==============================================================================
# Cell 1 — Dependency Installation (lm-evaluation-harness)
# ==============================================================================
# 1. Remove mismatched C++ extensions (torchvision & torchaudio) that crash transformers
# 2. Install lm_eval with typing_extensions >= 4.12.0
%uv pip uninstall torchvision torchaudio -q
%uv pip install -U "typing_extensions>=4.12.0" lm_eval[hf] accelerate transformers datasets huggingface_hub wandb -q

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Cell 2 — Hardware Probe & Environment Check

In [1]:
# ==============================================================================
# Cell 2 — Hardware Probe & Device Verification
# ==============================================================================
import torch
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Version    : {torch.version.cuda}")

# Probe GPU compute capability
p = torch.cuda.get_device_properties(0)
print(f"\nGPU Device     : {p.name}")
print(f"VRAM Capacity  : {p.total_memory / 1e9:.1f} GB")
print(f"Compute        : cc={p.major}.{p.minor}")
print(f"BFloat16       : {'Supported' if p.major >= 8 else 'NOT supported'}")

assert torch.cuda.is_available(), "No GPU detected!"
assert p.major >= 8, f"Requires Ampere+ GPU (A100/H100). Got cc={p.major}.{p.minor}"
print("\n✅ Hardware check passed for Modal L4")

PyTorch Version : 2.13.0+cu130
CUDA Version    : 13.0

GPU Device     : NVIDIA L4
VRAM Capacity  : 23.7 GB
Compute        : cc=8.9
BFloat16       : Supported

✅ Hardware check passed for Modal L4


## Cell 3 — Evaluation Configuration

In [2]:
# ==============================================================================
# Cell 3 — Authentication (Hugging Face & W&B) & Evaluation Configuration
# ==============================================================================
import os
try:
    from huggingface_hub import login, get_token
except ImportError:
    from huggingface_hub import login, HfFolder
    get_token = HfFolder.get_token

# Retrieve HF Token from all possible sources (os.environ, Colab Secrets, or cached token)
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN", "")
if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN_HERE":
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN") or userdata.get("HUGGINGFACE_TOKEN") or ""
    except Exception:
        pass

if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN_HERE":
    cached_token = get_token()
    if cached_token:
        HF_TOKEN = cached_token

if HF_TOKEN and HF_TOKEN != "YOUR_HF_TOKEN_HERE":
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    try:
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("✅ Authenticated with Hugging Face")
    except Exception as e:
        print(f"⚠️ Hugging Face authentication note: {e}")
else:
    print("ℹ️ HF_TOKEN not provided. Proceeding (public datasets/models remain accessible).")

# WandB Authentication
try:
    import wandb
    WANDB_TOKEN = os.environ.get("WANDB_API_KEY", "")
    if WANDB_TOKEN and WANDB_TOKEN != "YOUR_WANDB_KEY_HERE":
        wandb.login(key=WANDB_TOKEN, relogin=True)
        os.environ["WANDB_API_KEY"] = WANDB_TOKEN
        print("✅ Authenticated with Weights & Biases")
    else:
        print("ℹ️ WANDB_API_KEY not found. WandB tracking will operate in offline/disabled mode.")
        os.environ["WANDB_DISABLED"] = "true"
except ImportError:
    print("ℹ️ WandB module not installed. Operating without WandB tracking.")
    os.environ["WANDB_DISABLED"] = "true"

# ------------------------------------------------------------------------------
# Evaluation Configuration Parameters
# ------------------------------------------------------------------------------
HF_USERNAME = "abhinav0231"

# Target Model to Evaluate (Lily-1.5B trained with GRPO from Notebook 04)
MODEL_REPO = f"{HF_USERNAME}/Lily-1.5b-v0.1"

# Evaluation Benchmarks:
# - gsm8k: Grade-school math reasoning
# - arc_challenge: Multi-hop scientific reasoning
# - hellaswag: Common-sense reasoning & sentence completion
EVAL_TASKS  = "gsm8k,arc_challenge,hellaswag"
NUM_FEWSHOT = 8
BATCH_SIZE  = 16
OUTPUT_DIR  = "/root/lily_1_5b_eval_results"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"\nEvaluation Model : {MODEL_REPO}")
print(f"Benchmark Tasks  : {EVAL_TASKS}")
print(f"Few-Shot Setup   : {NUM_FEWSHOT}-shot")
print(f"Batch Size       : {BATCH_SIZE}")
print(f"Output Directory : {OUTPUT_DIR}")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ Authenticated with Hugging Face


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: abhinav0231 (abhinav0231-krmangalam) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ Authenticated with Weights & Biases

Evaluation Model : abhinav0231/Lily-1.5b-v0.1
Benchmark Tasks  : gsm8k,arc_challenge,hellaswag
Few-Shot Setup   : 8-shot
Batch Size       : 16
Output Directory : /root/lily_1_5b_eval_results


## Cell 4 — Run Benchmark Evaluation (`lm_eval`)

In [ ]:
# ==============================================================================
# Cell 4 — Launch lm-evaluation-harness CLI Subprocess
# ==============================================================================
import subprocess, sys

# Construct CLI command for lm_eval harness
cmd = [
    "lm_eval",
    "--model", "hf",
    "--model_args", f"pretrained={MODEL_REPO},dtype=bfloat16,trust_remote_code=True,attn_implementation=sdpa",
    "--tasks", EVAL_TASKS,
    "--num_fewshot", str(NUM_FEWSHOT),
    "--batch_size", str(BATCH_SIZE),
    "--apply_chat_template",
    "--output_path", OUTPUT_DIR,
    "--device", "cuda:0"
]

print(f"Launching benchmark evaluation suite...")
print(f"CMD: {' '.join(cmd)}\n")

# Stream output in real-time to avoid hidden tracebacks
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="", flush=True)

process.wait()
if process.returncode != 0:
    raise RuntimeError(f"Benchmark evaluation failed with exit code {process.returncode}")
print(f"\n✅ Benchmark evaluation finished successfully!")

Launching benchmark evaluation suite...
CMD: lm_eval --model hf --model_args pretrained=abhinav0231/Lily-1.5b-v0.1,dtype=bfloat16,trust_remote_code=True,attn_implementation=sdpa --tasks gsm8k,arc_challenge,hellaswag --num_fewshot 8 --batch_size 16 --apply_chat_template --output_path /root/lily_1_5b_eval_results --device cuda:0

2026-08-16:05:04:52 INFO     [config.evaluate_config:307] Using default fewshot_as_multiturn=True.
2026-08-16:05:05:08 INFO     [_cli.run:388] Selected Tasks: ['gsm8k', 'arc_challenge', 'hellaswag']
2026-08-16:05:05:11 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-08-16:05:05:11 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': 'abhinav0231/Lily-1.5b-v0.1', 'dtype': 'bfloat16', 'trust_remote_code': True, 'attn_implementation': 'sdpa'}
[RANK 0] Detected kernel version 4.19.0, which is below the recommended minimum of 5.5.0; 

## Cell 5 — Display & Parse Evaluation Summary

In [ ]:
# ==============================================================================
# Cell 5 — Parse Evaluation Results JSON & Display Accuracy Breakdown
# ==============================================================================
import glob, json

result_files = glob.glob(f"{OUTPUT_DIR}/**/*.json", recursive=True)
if result_files:
    latest_file = sorted(result_files)[-1]
    print(f"Parsing results file: {latest_file}\n")
    try:
        data = json.load(open(latest_file))
        results = data.get("results", {})
        print("="*50)
        print(f"  BENCHMARK RESULTS — {MODEL_REPO}")
        print("="*50)
        for task, metrics in results.items():
            acc = metrics.get("acc,none", metrics.get("acc", metrics.get("exact_match,none", "N/A")))
            if isinstance(acc, float):
                print(f"  {task:<20} : {acc*100:.2f}%")
            else:
                print(f"  {task:<20} : {acc}")
        print("="*50)
    except Exception as e:
        print(f"Could not parse results JSON: {e}")
else:
    print(f"Results saved in folder: {OUTPUT_DIR}")